# Halide Perovskite UV-Vis Optimization — Campaign Notebook

## Prerequisites

**Queue-server worker** must be running with the halide profile loaded:
```bash
XPD_HALIDE_CONFIG=/path/to/configs/halide_beamline.json \
XPD_HTTP_API_KEY=$QSERVER_HTTP_SERVER_API_KEY \
    start-re-manager --startup-module xpd_tools.optimization.profiles.halide
```

**Environment variables** (set in shell or `.env` before starting Jupyter):
```
QSERVER_HTTP_URI             queue-server HTTP endpoint
QSERVER_HTTP_SERVER_API_KEY  API key for the queue server
ZMQ_CONSUMER_ADDR            ZMQ document-dispatcher address
TILED_URI                    Tiled URI for raw/evaluated data
TILED_PROFILE                Tiled profile name (default: xpd)
MP_API_KEY                   Materials Project API key
```

## Usage modes

**Build fresh** (first run of a new campaign): run all cells top-to-bottom.
The config is archived to `configs/halide_beamline.json` for later reload.

**Reload prior work**: set `RELOAD_CONFIG = True` in the *Mode toggle* cell
below and set `RELOAD_CONFIG_PATH` to the JSON you want to resume from.
Skip to the *Build agent* cell — the `BuildAgent` is reconstructed from the
JSON so the exact same experiment parameters are used.

## Bring-up sequence

| Step | Notebook |
|------|----------|
| 1 | `validate_paths.ipynb` — device & PV connectivity (read-only) |
| 2 | `single_trial.ipynb` — one acquisition trial (pumps + QEPro) |
| 3 | **this notebook** — full optimization campaign |


## Environment Setup

In [ ]:
import os
import json
import numpy as np
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

# ── Queue server connection ───────────────────────────────────────────────
HTTP_SERVER_URI: str = os.environ.get(
    "QSERVER_HTTP_URI",
    "https://xf28id2-xpd-qs1.nsls2.bnl.gov",
)
HTTP_API_KEY: str = os.environ.get(
    "QSERVER_HTTP_SERVER_API_KEY",
    "",
)
ZMQ_CONSUMER_ADDR: str = os.environ.get(
    "ZMQ_CONSUMER_ADDR",
    "ipc:///var/lib/bluesky-zmq-proxy/xpd-ipc-in-ipc-out/out.sock",
)

# ── Tiled ─────────────────────────────────────────────────────────────────
TILED_URI: str = os.environ.get("TILED_URI", "https://tiled.nsls2.bnl.gov")
TILED_PROFILE: str = os.environ.get("TILED_PROFILE", "xpd")

# ── Historical data ───────────────────────────────────────────────────────
AGENT_DATA_PATH: str | None = "data/halide_data_uvvis_only.csv"
# AGENT_DATA_PATH = None  # comment-in to start from scratch

print("HTTP_SERVER_URI :", HTTP_SERVER_URI)
print("ZMQ_CONSUMER_ADDR:", ZMQ_CONSUMER_ADDR)
print("TILED_URI       :", TILED_URI)

## Mode Toggle — Build Fresh or Reload Prior Config

In [ ]:
# ── Set RELOAD_CONFIG = True to skip construction and reload a saved JSON ──
RELOAD_CONFIG: bool = False
RELOAD_CONFIG_PATH: str = "configs/halide_beamline.json"

if RELOAD_CONFIG:
    print(f"Mode: RELOAD  ←  {RELOAD_CONFIG_PATH}")
    print("Skip to the 'Build agent' cell below.")
else:
    print("Mode: BUILD FRESH — run all construction cells below.")

---
## AGENT — Initialization

*(Skip this section if `RELOAD_CONFIG = True` above.)*

Construct the `BuildAgent` with connection settings and the evaluation method.
Every parameter is spelled out explicitly here so it's visible in the notebook
and captured in the archived JSON.

In [ ]:
from xpd_tools.optimization.agent import BuildAgent

if not RELOAD_CONFIG:
    build_agent = BuildAgent(
        queue_server=True,
        agent_data_path=AGENT_DATA_PATH,
        http_server_uri=HTTP_SERVER_URI,
        http_api_key=HTTP_API_KEY,
        zmq_consumer_address=ZMQ_CONSUMER_ADDR,
        tiled_profile=TILED_PROFILE,
        evaluation_method="uvvis",
    )

    build_agent.set_metadata({
        "beamline": "28id2",
        "tags": ["qserver", "bluesky", "uvvis"],
        "comment": "Halide Perovskite UV-Vis Optimization",
    })

    print(build_agent.metadata_string)

## BEAMLINE — UV-Vis Objectives

In [ ]:
from xpd_tools.optimization.helpers.qepro import PlqyReference, SpectraFitSettings

# Optimization target
PEAK_TARGET: float = 520    # nm
PEAK_TOLERANCE: float = 5   # nm

if not RELOAD_CONFIG:
    build_agent.set_uvvis_objectives(
        peak_target=PEAK_TARGET,
        peak_tolerance=PEAK_TOLERANCE,
        fwhm_role="objective",
        plqy_role="objective",
        peak_distance_role="objective",
        max_peak_distance_nm=PEAK_TOLERANCE * 10,
        min_plqy=1e-3,
        max_fwhm_nm=60,
        max_retries=10,
        retry_delay=2.0,
        fit_settings=SpectraFitSettings(
            pl_screen_key_height=200,
            pl_screen_peak_height=50,
            pl_screen_peak_distance=100,
            pl_percent_range=(40, 100),
            pl_wavelength_range=(400, 800),
            pl_fit_maxfev=100000,
            pl_fit_r2_window_sigma=3,
            absorbance_percent_range=(10, 70),
            absorbance_wavelength_range=(210, 700),
        ),
        plqy=PlqyReference(
            excitation_wavelength_nm=365,
            absorbance=0.361,
            pl_integral=952628,
            refractive_index=1.337,
            plqy=0.546,
            solvent_refractive_index=1.506,
        ),
    )

    print(build_agent.fit_settings)
    print(build_agent.plqy)

## BLOP — Degrees of Freedom

Each `Pump.id` must match a key in the `devices` dict that the queue-server
worker built from `build_halide_devices()`.  The bounds are in ul/min.

In [ ]:
from xpd_tools.optimization.helpers.dofs import Pump

_dds2_p1 = "CsPb"
_dds2_p2 = "TOABr"
_dds3_p1 = "Cl"
_dds3_p2 = "I"

if not RELOAD_CONFIG:
    build_agent.set_dofs([
        Pump(name=_dds2_p1,   id="dds2_p1", bounds=(0, 100)),  # CsPb precursor
        Pump(name=_dds2_p2,   id="dds2_p2", bounds=(0, 100)),  # TOABr
        Pump(name=_dds3_p1,   id="dds3_p1", bounds=(0, 100)),  # TOACl / Pb-rich
        Pump(name=_dds3_p2,   id="dds3_p2", bounds=(0, 100)),  # TOAI
    ])

    print(build_agent.dofs)

## BEAMLINE — Experimental Hardware Parameters

`loaded_ml` is the volume (mL) physically loaded into each syringe on the day.
Update before each run; the plans check this to confirm enough reagent remains.

In [ ]:
from xpd_tools.optimization.plans import DilutionStage, FlowSource, WashCycle

if not RELOAD_CONFIG:
    build_agent.experiment(
        sources=[
            FlowSource(
                dof="infusion_rate_CsPb",
                pump="dds2_p1",
                precursor=_dds2_p1,
                sample_label="Cs-rich 33mM",
                loaded_ml=30.0,
            ),
            FlowSource(
                dof="infusion_rate_Br",
                pump="dds2_p2",
                precursor=_dds2_p2,
                sample_label="TOABr",
                loaded_ml=30.0,
            ),
            FlowSource(
                dof="infusion_rate_Cl",
                pump="dds3_p1",
                precursor=_dds3_p1,
                sample_label="Pb-rich 33mM",
                loaded_ml=30.0,
            ),
            FlowSource(
                dof="infusion_rate_I",
                pump="dds3_p2",
                precursor=_dds3_p2,
                sample_label="I",
                loaded_ml=30.0,
            ),
        ],
        dilutions=[
            DilutionStage(
                pump="dds1_p1",
                ratio=1.0,
                position="before_equilibrium",
                syringe_ml=20,
                material="plastic_BD",
                target_ml=20,
                loaded_ml=20.0,
            ),
            DilutionStage(
                pump="ultra2",
                ratio=1.0,
                position="after_equilibrium",
                syringe_ml=100,
                material="steel",
                target_ml=100,
                loaded_ml=100.0,
                wait_sec=30,
            ),
        ],
        wash_cycles=[
            WashCycle(pump="ultra1", loaded_ml=30.0),
        ],
    )

## BLOP — Run Settings

In [ ]:
if not RELOAD_CONFIG:
    build_agent.set_run(
        iterations=50,
        extra_initialization_trials=5,
        generation_strategy={
            "initialize_with_center": False,
            "min_observed_initialization_trials": 2,
            "allow_exceeding_initialization_budget": True,
        },
    )

    print(build_agent.run_settings)

## Archive Config / Reload Config

**Build-fresh path:** saves the config to `configs/halide_beamline.json`
(timestamped archive + stable copy) so it can be reloaded later.

**Reload path:** loads the prior JSON and reconstructs the `BuildAgent`.

In [ ]:
from xpd_tools.optimization.campaign import load_campaign_config, save_campaign_config

if RELOAD_CONFIG:
    build_agent = load_campaign_config(RELOAD_CONFIG_PATH)
    print("Reloaded build_agent:")
    print("  evaluation_method :", build_agent.evaluation_method)
    print("  DOFs              :", [p.id for p in build_agent.pumps])
    print("  run.iterations    :", build_agent.run_settings.iterations)
else:
    cfg_path = save_campaign_config(build_agent, "halide_beamline")
    print(f"Config archived → {cfg_path}")

---
## Build Agent (Queue-Server Path)

`build_agent.build()` returns a `QueueserverAgent` wired to the HTTP RE
Manager API and the ZMQ document dispatcher.  There is no local RunEngine
on this path — plans run on the worker.

In [ ]:
run_agent = build_agent.build()   # -> QueueserverAgent

print("QueueserverAgent built.")
print("  acquisition_plan:", run_agent.acquisition_plan)
print("  actuators       :", list(run_agent.actuators))

## Verify — Plan Registered on Worker

Confirms the worker profile loaded `uvvis_acquire` correctly.  If this cell
raises, check that the RE Manager is running and that `XPD_HALIDE_CONFIG`
is set on the worker process.

In [ ]:
from xpd_tools.optimization.campaign import assert_plan_registered

assert_plan_registered(run_agent, plan_name="uvvis_acquire")

## Configure Generation Strategy

In [ ]:
n_historical = build_agent.configure_generation_strategy(run_agent)
print(f"{n_historical} historical trials ingested into the generation strategy.")

---
## Optimization Loop

`run_agent.run(...)` returns immediately with a `Future`; the suggest →
acquire → ingest cycle runs asynchronously via ZMQ callbacks.  The
progress loop below prints the current iteration every 30 seconds so
you can see it advancing without blocking the kernel.

In [ ]:
import time

campaign_future = run_agent.run(
    iterations=build_agent.run_settings.iterations,
    n_points=build_agent.run_settings.n_points,
    checkpoint_interval=10,
)

print(f"Campaign started — {build_agent.run_settings.iterations} iterations.")
print("Watching progress (Ctrl-C or run_agent.stop() to halt early)...\n")

while not campaign_future.done():
    print(f"  iteration {run_agent.current_iteration} / {build_agent.run_settings.iterations}")
    time.sleep(30)

campaign_result = campaign_future.result()   # raises if any iteration errored
print("\nCampaign complete.")
print(campaign_result)

---
## Results — Summary and Band-Gap Plot

In [ ]:
import matplotlib.pyplot as plt

from xpd_tools.optimization.visualization import plot_bandgap_wavelength
from xpd_tools.optimization.campaign import summarize_campaign

df = summarize_campaign(run_agent)
df.head(10)

In [ ]:
%matplotlib inline

fig = plot_bandgap_wavelength(
    df,
    save="bandgap_wavelength.png",
)
plt.show()